In [ ]:
import pandas as pd

In [ ]:
from google.cloud import bigquery

client = bigquery.Client()

query = """
    SELECT 
      user_id,
      num_transactions,
      total_amount_usd,
      num_transaction_types,
      num_merchants,
      num_countries,
      pct_card_present,
      num_notifications,
      num_contacts,
      num_referrals,
      num_successful_referrals,
      age,
      age_group
    FROM `numeric-advice-452700-j9.neo_bank_.user_engagement`
"""

df_user_engagement = client.query(query).to_dataframe()

# Ahora df_user_engagement ya tiene todo listo, tal como salía del groupby:
print(df_user_engagement.head())

In [ ]:
# Asegúrate de que la fecha sea datetime
# df_user_engagement['create_date_notification'] = pd.to_datetime(df_user_engagement['create_date_notification'])

# Agrupa por usuario (suponiendo que hay una columna 'user_id')
user_engagement = df_user_engagement
user_engagement

In [ ]:
from sklearn.preprocessing import MinMaxScaler
# Selecciona las columnas a normalizar
cols_to_scale = [
    'num_transactions', 'total_amount_usd', 'num_transaction_types',
    'num_merchants', 'num_countries', 'pct_card_present',
    'num_notifications', 'num_contacts', 'num_referrals', 'num_successful_referrals'
]
scaler = MinMaxScaler()
user_engagement_scaled = user_engagement.copy()
user_engagement_scaled[cols_to_scale] = scaler.fit_transform(user_engagement[cols_to_scale])
user_engagement_scaled

In [ ]:
user_engagement_scaled['engagement_score'] = (
    0.2 * user_engagement_scaled['num_transactions'] +
    0.15 * user_engagement_scaled['total_amount_usd'] +
    0.1 * user_engagement_scaled['num_transaction_types'] +
    0.1 * user_engagement_scaled['num_merchants'] +
    0.05 * user_engagement_scaled['num_countries'] +
    0.1 * user_engagement_scaled['pct_card_present'] +
    0.1 * user_engagement_scaled['num_notifications'] +
    0.05 * user_engagement_scaled['num_contacts'] +
    0.1 * user_engagement_scaled['num_successful_referrals']
)

In [ ]:
# Define thresholds
high_threshold = user_engagement_scaled['engagement_score'].quantile(0.80)
low_threshold = user_engagement_scaled['engagement_score'].quantile(0.40)
# Assign engagement levels
user_engagement_scaled['engagement_level'] = user_engagement_scaled['engagement_score'].apply(
    lambda x: 'high' if x >= high_threshold else
              'medium' if x >= low_threshold else
              'low')

In [ ]:
user_engagement_scaled

In [ ]:
user_engagement_scaled.columns

In [ ]:
user_engagement_scaled.info()

In [ ]:
import plotly.express as px

# Generar tabla de frecuencias como DataFrame robusto
engagement_counts = (
    user_engagement_scaled['engagement_level']
    .value_counts(dropna=False)
    .rename_axis('engagement_level')
    .reset_index(name='num_users')
    .sort_values('engagement_level')  # opcional: orden alfabético
)

# Revisar la tabla
print(engagement_counts.head())

# Graficar
fig = px.bar(
    engagement_counts,
    x='engagement_level',
    y='num_users',
    text='num_users',
    color='engagement_level',
    color_discrete_sequence=px.colors.qualitative.Set2,
    labels={
        'engagement_level': 'Nivel de Engagement',
        'num_users': 'Número de Usuarios'
    },
    title='Distribución de usuarios por nivel de engagement'
)

fig.update_traces(textposition='outside')
fig.update_layout(
    xaxis_title='Nivel de Engagement',
    yaxis_title='Número de Usuarios',
    showlegend=False
)

fig.show()
